In [ ]:
import os
import re
from pathlib import Path
from math import floor
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
from rasterio.warp import transform as transform_coords

try:
    import rasterio
    from rasterio.warp import calculate_default_transform
    HAS_RASTERIO = True
except ImportError:
    HAS_RASTERIO = False

Image.MAX_IMAGE_PIXELS = None

# ==========================================
# 1. CONFIGURATION
# ==========================================

DOWNLOAD_DIRS = [
    Path(r"feburary_downloads"),
    Path(r"march_downloads"),
    Path(r"june_downloads"),
    Path(r"august_downloads"),
    Path(r"september_downloads"),
    Path(r"oktober_downloads"),
    Path(r"december_downloads")
]

OUTPUT_DIR = Path("results")
CROPS_DIR = OUTPUT_DIR / "crops_new"
BOX_SIZE = 100

MONTHLY_CONFIG = [
    {
        "month": "February",
        "flight_name": "UAL 909",
        "image_path": Path("feburary_images_azimuth_range/2025-02-04_trackbox_rawdoppler_rect.jpg"),
        "pois": [
            {
                "id": "POI_1",
                "corners": [(18915, 4036), (18918, 4036), (18915, 4040), (18918, 4040)]
            }
        ],
        "plane_pos1": (19470, 1683),
        "plane_pos2": (18826, 1779)
    },

    {
        "month": "June",
        "flight_name": "AAL730",
        "image_path": Path("june_images_azimuth_range/2025-06-22_trackbox_rawdoppler_rect.jpg"),
        "pois": [
            {
                "id": "POI_1",
                "corners": [(19755, 6797), (19756, 6799), (19756, 6799), (19756, 6797)]
            },

        ],
        "plane_pos1": (20283, 4197),
        "plane_pos2": (19664, 4536)
    },

    {
        "month": "August",
        "flight_name": "VIR110",
        "image_path": Path("august_images_azimuth_range/2025-08-21_trackbox_rawdoppler_rect.jpg"),
        "pois": [
            {
                "id": "POI_1",
                "corners": [(21891, 16438), (21891, 16442), (21894, 16442), (21894, 16438)]
            },
        ],
        "plane_pos1": (21219, 14124),
        "plane_pos2": (21834, 14180)
    },

    {
        "month": "September",
        "flight_name": "BAW14k",
        "image_path": Path("september_images_azimuth_range/2025-09-20_trackbox_rawdoppler_rect.jpg"),
        "pois": [
            {
                "id": "POI_1",
                "corners": [(20947, 6623), (20947, 6649), (20951, 6623), (20951, 6649)]
            },
        ],
        "plane_pos1": (20685, 4403),
        "plane_pos2": (20039, 4458)
    },

    {
        "month": "September",
        "flight_name": "DAL176",
        "image_path": Path("september_images_azimuth_range/2025-09-20_trackbox_rawdoppler_rect.jpg"),
        "pois": [
            {
                "id": "POI_1",
                "corners": [(24770, 13153), (24770, 13156), (24771, 13156), (24771, 13153)]
            },
        ],
        "plane_pos1": (24434, 10851),
        "plane_pos2": (23806, 10782)
    },

    {
        "month": "Oktober",
        "flight_name": "UAL953",
        "image_path": Path("oktober_images_azimuth_range/2025-10-20_trackbox_rawdoppler_rect.jpg"),
        "pois": [
            {
                "id": "POI_1",
                "corners": [(13744, 6012), (13747, 6017), (13744, 6017), (13747, 6012)]
            },
        ],
        "plane_pos1": (13762, 4503),
        "plane_pos2": (13110, 4693)
    },

    {
        "month": "Oktober",
        "flight_name": "UAL987",
        "image_path": Path("oktober_images_azimuth_range/2025-10-26_trackbox_rawdoppler_rect.jpg"),
        "pois": [
            {
                "id": "POI_1",
                "corners": [(23059, 8540), (23059, 8543), (23062, 8540), (23062, 8543)]
            },
        ],
        "plane_pos1": (23594, 5919),
        "plane_pos2": (22889, 6034)
    },

    {
        "month": "December",
        "flight_name": "DAL162",
        "image_path": Path("december_images_azimuth_range/2025-12-13_trackbox_rawdoppler_rect.jpg"),
        "pois": [
            {
                "id": "POI_1",
                "corners": [(20848, 6587), (20848, 6590), (20852, 6587), (20852, 6590)]
            },
            {
                "id": "POI_2",
                "corners": [(20859, 6641), (20859, 6651), (20863, 6641), (20863, 6651)]
            },
        ],
        "plane_pos1": (21082, 4375),
        "plane_pos2": (20367, 4435)
    },
]

# ==========================================
# 2. HELPER FUNCTIONS
# ==========================================

def find_date_in_filename(path):
    """Extract YYYY-MM-DD date string from image filename."""
    match = re.search(r"(\d{4})-(\d{2})-(\d{2})", Path(path).name)
    if match is None:
        return "N/A"
    return match.group(0)

def find_matching_tiff(image_path):
    """Locate matching original GeoTIFF file in download directories based on date."""
    date_code = find_date_in_filename(image_path).replace("-", "")
    if date_code == "N/A":
        return None

    for folder in DOWNLOAD_DIRS:
        folder = Path.cwd() / folder if not folder.is_absolute() else folder
        if not folder.exists():
            continue
        matches = sorted(folder.rglob(f"*{date_code}*.tiff"))
        if matches:
            return matches[0]
    return None

def transform_pixel_to_latlon(pixel_x, pixel_y, tiff_path):
    """
    Transforms given (pixel_x, pixel_y) pixel coordinates to (latitude, longitude)
    using the native transform and reprojecting to EPSG:4326.
    """
    if not HAS_RASTERIO or tiff_path is None or not Path(tiff_path).exists():
        return np.nan, np.nan

    with rasterio.open(tiff_path) as src:
        if src.crs is not None and src.transform != rasterio.transform.IDENTITY:
            x_native, y_native = src.transform * (pixel_x, pixel_y)
            source_crs = src.crs
            
        elif src.gcps and src.gcps[0]: 
            gcps, gcp_crs = src.gcps
            gcp_transform = rasterio.transform.from_gcps(gcps)
            x_native, y_native = gcp_transform * (pixel_x, pixel_y)
            source_crs = gcp_crs
            
        else:
            print(f"Warning: No valid CRS or GCPs found in {tiff_path}")
            return np.nan, np.nan

        lons, lats = transform_coords(
            source_crs, 
            "EPSG:4326", 
            [x_native], 
            [y_native]
        )
        
        return lats[0], lons[0]

def generate_pixel_grid_from_corners(corners):
    """Generates all integer pixel tuples within the bounding box of 4 corners."""
    xs = [int(floor(pt[0])) for pt in corners]
    ys = [int(floor(pt[1])) for pt in corners]
    xmin, xmax = min(xs), max(xs)
    ymin, ymax = min(ys), max(ys)
    
    pixel_set = [
        (x, y) 
        for x in range(xmin, xmax + 1) 
        for y in range(ymin, ymax + 1)
    ]
    return pixel_set

def compute_cutout_and_stats(img_array, pixels, box_size=100):
    """Crops a cutout box around target pixels and computes statistical metrics."""
    from math import erfc, sqrt

    img_h, img_w = img_array.shape
    xs = [p[0] for p in pixels]
    ys = [p[1] for p in pixels]
    cx = int(round(np.mean(xs)))
    cy = int(round(np.mean(ys)))
    half_box = box_size // 2

    crop_box = (
        max(0, cx - half_box),
        max(0, cy - half_box),
        min(img_w, cx + half_box),
        min(img_h, cy + half_box),
    )
    x0, y0, x_stop, y_stop = crop_box
    cutout = img_array[y0:y_stop, x0:x_stop]

    mask = np.ones(cutout.shape, dtype=bool)
    source_values = []
    for x, y in pixels:
        sx, sy = x - x0, y - y0
        if 0 <= sx < cutout.shape[1] and 0 <= sy < cutout.shape[0]:
            source_values.append(cutout[sy, sx])
            mask[sy, sx] = False

    if not source_values:
        source_values = [cutout[cutout.shape[0] // 2, cutout.shape[1] // 2]]

    source = np.array(source_values)
    background = cutout[mask] if np.any(mask) else cutout.flatten()

    source_mean = float(source.mean())
    source_max = float(source.max())
    bg_mean = float(background.mean())
    bg_std = float(background.std(ddof=1)) if len(background) > 1 else 0.0

    sigma_mean = (source_mean - bg_mean) / bg_std if bg_std > 0 else 0.0
    sigma_max = (source_max - bg_mean) / bg_std if bg_std > 0 else 0.0

    p_mean = float(0.5 * erfc(sigma_mean / sqrt(2)))
    p_max = float(0.5 * erfc(sigma_max / sqrt(2)))

    stats = {
        "source_mean": source_mean,
        "source_max": source_max,
        "bg_mean": bg_mean,
        "bg_std": bg_std,
        "sigma_mean": sigma_mean,
        "sigma_max": sigma_max,
        "p_mean": p_mean,
        "p_max": p_max,
    }
    return cutout, crop_box, stats

def save_cutout_image(cutout, highlight_pixels, crop_box, output_filename, header_text=""):
    """
    Saves a cutout image with padding ONLY at the top edge for header text.
    """
    x0, y0, _, _ = crop_box
    
    fig, ax = plt.subplots(figsize=(5, 6.5), dpi=150)
    
    ax.imshow(
        cutout, 
        cmap="gray", 
        vmin=np.percentile(cutout, 2), 
        vmax=np.percentile(cutout, 98)
    )

    if highlight_pixels:
        for x, y in highlight_pixels:
            ax.scatter([x - x0], [y - y0], c="red", marker="x", s=80)

    ax.axis("off")
    ax.set_frame_on(False)
    
    if header_text:
        ax.set_title(f"\n{header_text}", fontsize=14, loc="center", pad=12, fontweight="bold", linespacing=1.3)

    plt.tight_layout()
    plt.savefig(
        output_filename, 
        bbox_inches="tight", 
        pad_inches=0, 
        edgecolor="none"
    )
    plt.close(fig)

# ==========================================
# 3. EXECUTION PIPELINE
# ==========================================

def run_pipeline():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    CROPS_DIR.mkdir(parents=True, exist_ok=True)

    records = []

    for item in MONTHLY_CONFIG:
        month = item["month"]
        flight_name = item.get("flight_name", "Unknown Flight")
        
        flight_slug = re.sub(r"[^a-zA-Z0-9]", "_", flight_name.lower())

        img_path = Path.cwd() / item["image_path"] if not item["image_path"].is_absolute() else item["image_path"]
        print(f"--- Processing: {month} ({flight_name} - {img_path.name}) ---")

        if not img_path.exists():
            print(f"Error: Image file not found: {img_path}")
            continue

        with Image.open(img_path) as img:
            img_gray = img.convert("L")
            img_array = np.asarray(img_gray, dtype=float)

        tiff_path = find_matching_tiff(img_path)
        print(f"Matching GeoTIFF: {tiff_path}")

        date_str = find_date_in_filename(img_path)

        # ----------------------------------
        # PROCESS POINTS OF INTEREST (POIs)
        # ----------------------------------
        for poi in item["pois"]:
            poi_id = poi["id"]
            corners = poi["corners"]

            pixels = generate_pixel_grid_from_corners(corners)
            cutout, crop_box, stats = compute_cutout_and_stats(img_array, pixels, BOX_SIZE)

            center_x = np.mean([x for x, y in pixels]) + 0.5
            center_y = np.mean([y for x, y in pixels]) + 0.5

            lat, lon = transform_pixel_to_latlon(center_x, center_y, tiff_path)

            # Transform each corner of the polygon to lat/lon coordinates
            corners_latlon = []
            for cx, cy in corners:
                corner_lat, corner_lon = transform_pixel_to_latlon(cx, cy, tiff_path)
                corners_latlon.append((round(corner_lat, 6), round(corner_lon, 6)) if not np.isnan(corner_lat) else (np.nan, np.nan))

            # header_text = (
            #     f"{flight_name:^100}\n"
            #     f"{date_str:^100}"
            # )

            mosaic_num = MONTHLY_CONFIG.index(item) + 1
            marked_filename = f"Mosaic{mosaic_num}({flight_name})_marked.png"

            marked_filepath = CROPS_DIR / marked_filename
            save_cutout_image(cutout, pixels, crop_box, marked_filepath) #, header_text)

            clean_filename = f"Mosaic{mosaic_num}({flight_name}).png"
            clean_filepath = CROPS_DIR / clean_filename
            save_cutout_image(cutout, [], crop_box, clean_filepath) #, header_text)

            records.append({
                "Month/File": month,
                "Flight Name": flight_name,
                "Source Image": img_path.name,
                "Object Type": f"POI ({poi_id})",
                "POI Corners (px)": str(corners),
                "POI Corners (lat/lon)": str(corners_latlon),  # Added polygon corners in lat/lon
                "Crop Marked Filename": str(marked_filepath.relative_to(OUTPUT_DIR)),
                "Crop Clean Filename": str(clean_filepath.relative_to(OUTPUT_DIR)),
                "Pixel X": center_x - 0.5,
                "Pixel Y": center_y - 0.5,
                "Latitude": lat,
                "Longitude": lon,
                "Source Mean": stats["source_mean"],
                "Source Max": stats["source_max"],
                "Background Mean": stats["bg_mean"],
                "Background Std": stats["bg_std"],
                "Sigma Max": stats["sigma_max"],
                "P-Value Max": stats["p_max"],
            })

        # ----------------------------------
        # PROCESS AIRCRAFT POSITIONS (1 & 2)
        # ----------------------------------
        aircraft_positions = [
            ("Aircraft (Plane Pos 1)", item["plane_pos1"]),
            ("Aircraft (Plane Pos 2)", item["plane_pos2"]),
        ]

        for plane_label, plane_pos in aircraft_positions:
            px, py = plane_pos
            lat, lon = transform_pixel_to_latlon(px + 0.5, py + 0.5, tiff_path)

            records.append({
                "Month/File": month,
                "Flight Name": flight_name,
                "Source Image": img_path.name,
                "Object Type": plane_label,
                "POI Corners (px)": "N/A",
                "POI Corners (lat/lon)": "N/A",  # N/A for aircraft records
                "Crop Marked Filename": "N/A",
                "Crop Clean Filename": "N/A",
                "Pixel X": px,
                "Pixel Y": py,
                "Latitude": lat,
                "Longitude": lon,
                "Source Mean": np.nan,
                "Source Max": np.nan,
                "Background Mean": np.nan,
                "Background Std": np.nan,
                "Sigma Max": np.nan,
                "P-Value Max": np.nan,
            })

    # Save Results
    df = pd.DataFrame(records)
    csv_path = OUTPUT_DIR / "poi_analysis_results.csv"
    excel_path = OUTPUT_DIR / "poi_analysis_results.xlsx"

    df.to_csv(csv_path, index=False)
    df.to_excel(excel_path, index=False)  # Optional: saving to excel as well

    print("\n--- Processing Complete ---")
    print(f"Results CSV saved to: {csv_path.resolve()}")
    print("\nSummary Output:")
    print(df[["Flight Name", "Object Type", "POI Corners (px)", "POI Corners (lat/lon)", "Latitude", "Longitude"]].to_string())
    return df

if __name__ == "__main__":
    df_results = run_pipeline()

--- Processing: February (UAL 909 - 2025-02-04_trackbox_rawdoppler_rect.jpg) ---
Matching GeoTIFF: c:\Users\Johannes Pielmeier\OneDrive\Education\DTU 2024 - 2029\Fagprojekt\2025\feburary_downloads\S1A_IW_GRDH_1SDV_20250204T065541_20250204T065606_057743_071E5B_2620.SAFE\s1a-iw-grd-vv-20250204t065541-20250204t065606-057743-071e5b-001.tiff
--- Processing: June (AAL730 - 2025-06-22_trackbox_rawdoppler_rect.jpg) ---
Matching GeoTIFF: c:\Users\Johannes Pielmeier\OneDrive\Education\DTU 2024 - 2029\Fagprojekt\2025\june_downloads\S1C_IW_GRDH_1SDV_20250622T065426_20250622T065451_002892_005EAC_4532.SAFE\s1c-iw-grd-vv-20250622t065426-20250622t065451-002892-005eac-001.tiff
--- Processing: August (VIR110 - 2025-08-21_trackbox_rawdoppler_rect.jpg) ---
Matching GeoTIFF: c:\Users\Johannes Pielmeier\OneDrive\Education\DTU 2024 - 2029\Fagprojekt\2025\august_downloads\S1C_IW_GRDH_1SDV_20250821T065430_20250821T065455_003767_00784B_CE75.SAFE\s1c-iw-grd-vv-20250821t065430-20250821t065455-003767-00784b-001.ti